# 02 — Strategy Agent Tests

Validates the Strategist pipeline:
1. Knowledge-base tools (civs, units, counters, buildings)
2. `build_strategist` Deep Agent construction
3. Single-turn strategy invocation on a synthetic `GameStateSnapshot`
4. End-to-end trace: snapshot → strategy text

> **No game screen required** — all tests run against the KB JSON files and Ollama.

In [1]:
import sys
sys.path.insert(0, "/mnt/e/Personal/Samarth/repository/AOE-Agent/src")

from aoe2_agent.schemas import GameStateSnapshot, DetectedUnit
print("Imports OK")

Imports OK


## 1 — Knowledge Base Tools

In [2]:
from aoe2_agent.knowledge.tools import (
    get_civilization_info,
    get_unit_counters,
    get_unit_stats,
    get_building_info,
    KB_TOOLS,
)

print(f"{len(KB_TOOLS)} tools registered: {[f.__name__ for f in KB_TOOLS]}")

4 tools registered: ['get_civilization_info', 'get_unit_counters', 'get_unit_stats', 'get_building_info']


In [3]:
# Civilization lookup
import json

result = get_civilization_info("Mongols")
civ = json.loads(result)
print(f"Civilization: {civ['name']}")
print("Bonuses:")
for b in civ.get('bonuses', []):
    print(f"  • {b}")

Civilization: Mongols
Bonuses:
  • Hunters work +40% faster
  • Cavalry Archers attack +25% faster
  • Light Horsemen +20/30% HP in Castle/Imperial Age


In [4]:
# Exact match
print("--- Britons ---")
result = get_civilization_info("Britons")
civ = json.loads(result)
print(f"Name: {civ['name']}")
print(f"First bonus: {civ['bonuses'][0]}")

# Case-insensitive partial match
print("\n--- 'frank' (partial) ---")
print(get_civilization_info("frank")[:200], "...")

# Not found
print("\n--- Unknown civ ---")
print(get_civilization_info("Klingons"))

--- Britons ---
Name: Britons
First bonus: Shepherds work +25% faster

--- 'frank' (partial) ---
{
  "name": "Franks",
  "key": "Franks",
  "bonuses": [
    "Foragers work +15% faster",
    "Mill technologies free",
    "Mounted Units +20% HP starting in Feudal Age",
    "Castles cost -15/25% in  ...

--- Unknown civ ---
Civilization 'Klingons' not found. Available: ['Armenians', 'Aztecs', 'Bengalis', 'Berbers', 'Bohemians', 'Britons', 'Bulgarians', 'Burgundians', 'Burmese', 'Byzantines', 'Celts', 'Chinese', 'Cumans', 'Dravidians', 'Ethiopians', 'Franks', 'Georgians', 'Goths', 'Gurjaras', 'Hindustanis', 'Huns', 'Inca', 'Italians', 'Japanese', 'Jurchens', 'Khitans', 'Khmer', 'Koreans', 'Lithuanians', 'Magyars', 'Malay', 'Malians', 'Maya', 'Mongols', 'Persians', 'Poles', 'Portuguese', 'Romans', 'Saracens', 'Shu', 'Sicilians', 'Slavs', 'Spanish', 'Tatars', 'Teutons', 'Turks', 'Vietnamese', 'Vikings', 'Wei', 'Wu']


In [5]:
# Unit counter lookup
for unit_type in ["archer", "cavalry", "knight", "monk"]:
    result = json.loads(get_unit_counters(unit_type))
    data = result[unit_type]
    print(f"{unit_type:15s} → strong vs {data['strong_vs'][:2]}  |  weak vs {data['weak_vs'][:2]}")

archer          → strong vs ['infantry', 'spearman']  |  weak vs ['skirmisher', 'mangonel']
cavalry         → strong vs ['archer', 'siege']  |  weak vs ['spearman', 'camel']
knight          → strong vs ['archer', 'siege']  |  weak vs ['pikeman', 'camel']
monk            → strong vs ['knight', 'elephant']  |  weak vs ['light_cavalry', 'eagle_warrior']


In [6]:
# Unit stats lookup
for unit_name in ["Archer", "Knight", "Hussar", "Trebuchet"]:
    result = json.loads(get_unit_stats(unit_name))
    cost = result.get('cost', {})
    print(
        f"{result['name']:20s}  HP={result['hit_points']:4}  "
        f"ATK={result['attack']:3}  Cost: F{cost.get('food',0)} W{cost.get('wood',0)} G{cost.get('gold',0)}"
    )

Archer                HP=  30  ATK=  4  Cost: F0 W25 G45
Knight                HP= 100  ATK= 10  Cost: F60 W0 G75
Hussar                HP=  75  ATK=  7  Cost: F80 W0 G0
Mounted Trebuchet     HP=  75  ATK= 30  Cost: F175 W0 G175


In [7]:
# Building lookup
for bname in ["Town Center", "Castle", "Archery Range", "Stable"]:
    result = json.loads(get_building_info(bname))
    cost = result.get('cost', {})
    print(
        f"{result['name']:20s}  HP={result['hit_points']:5}  "
        f"Category={result['category']:12s}  Cost: W{cost.get('wood',0)} S{cost.get('stone',0)}"
    )

Town Center           HP= 2400  Category=economic      Cost: W275 S100
Castle                HP= 4800  Category=military      Cost: W0 S650
Archery Range         HP= 1800  Category=military      Cost: W175 S0
Stable                HP= 1800  Category=military      Cost: W175 S0


## 2 — Build the Strategist Deep Agent

In [8]:
from aoe2_agent.strategy.agent import build_strategist

PLAYER_CIV   = "Mongols"
OPPONENT_CIV = "Vietnamese"   # ← opponent civ now required

agent = build_strategist(PLAYER_CIV, OPPONENT_CIV)
print(f"Strategist agent built for {PLAYER_CIV} vs {OPPONENT_CIV}")
print(type(agent))

Strategist agent built for Mongols
<class 'langgraph.graph.state.CompiledStateGraph'>


## 3 — Strategy Invocation on Synthetic Snapshots

In [9]:
from aoe2_agent.strategy.agent import _snapshot_to_prompt
from collections import deque

# Dark Age — opening scenario
dark_snap = GameStateSnapshot(
    age="Dark",
    population=6,
    pop_cap=10,
    resources={"food": 400, "wood": 250, "gold": 0, "stone": 0},
    visible_units=[DetectedUnit(label="villager", owner="self", approx_count=6)],
    visible_buildings=["Town Center", "House"],
    idle_villagers_visible=False,
    threat_level="none",
    notes="Game just started",
)

# Prompt now includes both civilizations
prompt = _snapshot_to_prompt(dark_snap, deque(), PLAYER_CIV, OPPONENT_CIV)
print("--- Prompt sent to Strategist ---")
print(prompt)

--- Prompt sent to Strategist ---
Current game state for Mongols:

Age: Dark
Population: 6/10
Resources: food=400, wood=250, gold=0, stone=0
Idle villagers: False
Threat level: none
Visible units: 6x villager (self)
Visible buildings: Town Center, House
Notes: Game just started

Please analyze and provide your strategic advice.



In [10]:
# Invoke the Strategist on the Dark Age scenario
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result)
messages = result.get("messages", [])
strategy_text = ""
for msg in reversed(messages):
    content = getattr(msg, "content", None) or msg.get("content", "")
    if content:
        strategy_text = content
        break

print(strategy_text)

{'messages': [HumanMessage(content='Current game state for Mongols:\n\nAge: Dark\nPopulation: 6/10\nResources: food=400, wood=250, gold=0, stone=0\nIdle villagers: False\nThreat level: none\nVisible units: 6x villager (self)\nVisible buildings: Town Center, House\nNotes: Game just started\n\nPlease analyze and provide your strategic advice.\n', additional_kwargs={}, response_metadata={}, id='d3f9470e-2253-428e-8e67-2b0d8b099873'), AIMessage(content="Based on the current game state for Mongols:\n\n* Age: Dark - This is a good starting age as it provides access to all basic units and buildings.\n* Population: 6/10 - With 6 villagers, you have a decent population to start with. Try to balance resource gathering and building construction to increase your population quickly.\n* Resources: food=400, wood=250, gold=0, stone=0 - You have an abundance of food, which is great for expansion and unit production. Focus on gathering more wood to build essential buildings like the Town Center, Archer

In [ ]:
# Castle Age — under threat scenario
threat_snap = GameStateSnapshot(
    age="Castle",
    population=95,
    pop_cap=100,
    resources={"food": 150, "wood": 200, "gold": 80, "stone": 0},
    visible_units=[
        DetectedUnit(label="knight", owner="enemy", approx_count=8),
        DetectedUnit(label="villager", owner="self", approx_count=12),
        DetectedUnit(label="spearman", owner="self", approx_count=4),
    ],
    visible_buildings=["Town Center", "Castle", "Stable", "Barracks"],
    idle_villagers_visible=True,
    threat_level="high",
    notes="8 enemy knights approaching Town Center from the east",
)

prompt2 = _snapshot_to_prompt(threat_snap, deque(), PLAYER_CIV, OPPONENT_CIV)
result2 = agent.invoke({"messages": [{"role": "user", "content": prompt2}]})

messages2 = result2.get("messages", [])
strategy2 = ""
for msg in reversed(messages2):
    content = getattr(msg, "content", None) or msg.get("content", "")
    if content:
        strategy2 = content
        break

print(strategy2)

In [ ]:
# Imperial Age — late game push
imperial_snap = GameStateSnapshot(
    age="Imperial",
    population=180,
    pop_cap=200,
    resources={"food": 1200, "wood": 800, "gold": 500, "stone": 300},
    visible_units=[
        DetectedUnit(label="trebuchet", owner="self", approx_count=3),
        DetectedUnit(label="cavalry archer", owner="self", approx_count=20),
        DetectedUnit(label="halberdier", owner="enemy", approx_count=10),
    ],
    visible_buildings=["Town Center", "Castle", "Siege Workshop", "University"],
    idle_villagers_visible=False,
    threat_level="low",
    notes="Pushing enemy base; enemy has halberdiers defending",
)

prompt3 = _snapshot_to_prompt(imperial_snap, deque(), PLAYER_CIV, OPPONENT_CIV)
result3 = agent.invoke({"messages": [{"role": "user", "content": prompt3}]})

messages3 = result3.get("messages", [])
strategy3 = ""
for msg in reversed(messages3):
    content = getattr(msg, "content", None) or msg.get("content", "")
    if content:
        strategy3 = content
        break

print(strategy3)

## 4 — Direct KB Tool Calls Inside Agent

Verify the agent actually uses its tools (KB grounding per Article VII).

In [ ]:
# Ask the agent a pure knowledge question — it must call get_civilization_info
kb_result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "What are the Mongols' civilization bonuses? "
            "Use your get_civilization_info tool and list them."
        )
    }]
})

msgs = kb_result.get("messages", [])
for msg in reversed(msgs):
    content = getattr(msg, "content", None) or msg.get("content", "")
    if content:
        print(content)
        break

In [ ]:
# Counter question: what beats knights?
counter_result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "The enemy has 8 knights charging at me. "
            "Use get_unit_counters to find what beats them, "
            "then use get_unit_stats to compare costs."
        )
    }]
})

msgs = counter_result.get("messages", [])
for msg in reversed(msgs):
    content = getattr(msg, "content", None) or msg.get("content", "")
    if content:
        print(content)
        break

## 5 — StrategistLoop (async, single tick)

In [ ]:
import asyncio
from collections import deque
from aoe2_agent.strategy.agent import _snapshot_to_prompt

strategy_updates = []

# Simulate what StrategistLoop does internally for one tick
snap = dark_snap  # re-use Dark Age snapshot from above
history = deque()

prompt_text = _snapshot_to_prompt(snap, history, PLAYER_CIV, OPPONENT_CIV)

def run_one_tick():
    res = agent.invoke({"messages": [{"role": "user", "content": prompt_text}]})
    msgs = res.get("messages", [])
    for msg in reversed(msgs):
        content = getattr(msg, "content", None) or msg.get("content", "")
        if content:
            return content
    return ""

# Run in a thread (mirrors StrategistLoop.run's asyncio.to_thread call)
update = await asyncio.to_thread(run_one_tick)
strategy_updates.append(update)

print("--- Strategy update ---")
print(update)